In [52]:
import logging
import requests
# from daemon_client import send_to_daemon, ping, load_fernet
# from validate_credentials import validate_credentials
# from send_link import send_link
# from remote_desktop import transmit

log = logging.getLogger(__name__)

DEFAULT_TIMEOUT = 10  # seconds

# ---------------------------
logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

# Minimum required fields for a transmittable client package
REQUIRED_CLIENT_PKG_FIELDS = {"key", "endpoint", "transport"}

ModuleNotFoundError: No module named 'daemon_client'

In [39]:
def validate_credentials(origin: dict, a: dict) -> dict | bool:
    return True

In [40]:

def send_link(process: dict) -> bool:
    """
    Establish a landing zone by posting a link/handshake to the target endpoint.

    Expects process to contain:
        client_pkg.endpoint  — target URL (webapp) or IP (ssh)
        client_pkg.key       — encrypted credential bytes

    Returns:
        True if endpoint acknowledged, False otherwise
    """
    client_pkg = process.get("client_pkg", {})
    endpoint   = client_pkg.get("endpoint")
    key        = client_pkg.get("key")

    if not endpoint or not key:
        log.error("send_link: missing 'endpoint' or 'key' in client_pkg")
        return False

    payload = {
        "handshake": True,
        "key_hash":  _safe_hash(key),   # never send raw key in handshake
    }
    
    try:
        response = requests.post(
            f"{endpoint}/landing",
            json=payload,
            timeout=DEFAULT_TIMEOUT,
        )
        response.raise_for_status()
        log.info(f"Landing zone established at {endpoint} — status {response.status_code}")
        return True

    except requests.exceptions.ConnectionError:
        log.error(f"send_link: could not reach {endpoint}")
    except requests.exceptions.Timeout:
        log.error(f"send_link: timed out after {DEFAULT_TIMEOUT}s")
    except requests.exceptions.HTTPError as e:
        log.error(f"send_link: server returned error: {e}")

    return False
    

def _safe_hash(value) -> str:
    """Return a SHA-256 hex digest of the value for handshake verification."""
    import hashlib
    raw = value if isinstance(value, bytes) else str(value).encode()
    return hashlib.sha256(raw).hexdigest()

In [41]:

def build_client_pkg(process: dict) -> dict:
    """
    Extract and validate the minimum transmit signature from a process dict.
    Raises KeyError if required fields are missing.
    """
    pkg = process.get("client_pkg", {})
    missing = REQUIRED_CLIENT_PKG_FIELDS - pkg.keys()
    if missing:
        raise KeyError(f"client_pkg missing required fields: {missing}")
    return pkg



In [42]:

def transmit(pkg):

    return True

In [1]:
import threading
import socket
import json
import time
import hashlib
from cryptography.fernet import Fernet

# --- fake dependencies ---
def build_client_pkg(process):
    return process["client_pkg"]

def validate_credentials(origin, client_pkg):
    return origin.get("key") == client_pkg.get("key")

def send_link(process):
    print("send_link called")

def transmit(client_pkg):
    print("transmit called")

def load_fernet(path):
    return Fernet(Fernet.generate_key())

In [8]:
def send_credential(process: dict, origin: dict) -> bool:
    """
    Validate credentials, establish landing zone, transmit credentials.

    Transport options in client_pkg:
        "webapp"  — HTTP POST via Flask server (remote_desktop.py)
        "ssh"     — remote bash script via OpenSSH
        "daemon"  — direct TCP socket to credential daemon (ssh_daemon.py port 9922)

    Args:
        process: must contain 'client_pkg' with keys: endpoint, key, transport
        origin:  reference credential dict for validation

    Returns:
        True on successful transmission, False otherwise
    """
    try:
        client_pkg = build_client_pkg(process)
    except KeyError as e:
        log.error(f"Malformed process package: {e}")
        return False

    if not validate_credentials(origin, client_pkg):
        log.warning("Credential validation failed — aborting transmission")
        return False

    transport = client_pkg.get("transport")

    try:
        # ── Webapp transport ──────────────────────────────────────────────
        if transport == "webapp":
            log.info("Establishing webapp landing zone...")
            send_link(process)
            transmit(client_pkg)

        # ── SSH transport ─────────────────────────────────────────────────
        elif transport == "ssh":
            from ssh import ssh
            username = client_pkg.get("username")
            ip       = client_pkg.get("endpoint")
            if not username or not ip:
                raise ValueError("SSH transport requires 'username' and 'endpoint' in client_pkg")
            log.info(f"Initiating SSH transmission to {username}@{ip}...")
            ssh(username, ip)

        # ── Daemon transport ──────────────────────────────────────────────
        elif transport == "daemon":
            host      = client_pkg.get("endpoint")
            port      = int(client_pkg.get("daemon_port", 9922))
            key_path  = client_pkg.get("fernet_key_path", "fernet.key")

            if not host:
                raise ValueError("Daemon transport requires 'endpoint' in client_pkg")

            fernet = load_fernet(key_path)

            # Confirm daemon is alive before attempting delivery
            if not ping(host, port):
                raise ConnectionRefusedError(
                    f"Daemon not responding at {host}:{port} — "
                    "check that credential-daemon.service is running on the target machine"
                )

            result = send_to_daemon(
                credential=client_pkg,
                host=host,
                fernet=fernet,
                port=port,
            )
            log.info(f"Stored in daemon ledger — block #{result['block_index']}")

        else:
            raise ValueError(
                f"Unknown transport: '{transport}' — expected 'webapp', 'ssh', or 'daemon'"
            )

        log.info("Transmission complete.")
        return True

    except Exception as e:
        log.error(f"Transmission failed: {e}")
        return False

In [9]:

#send credential test stubs
def fake_daemon(host, port):
    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind((host, port))
    srv.listen(1)

    conn, _ = srv.accept()
    data = conn.recv(8192)

    msg = json.loads(data.decode().strip())

    response = {
        "status": "ok",
        "block_index": 7,
        "block_hash": hashlib.sha256(b"block").hexdigest()
    }

    conn.sendall((json.dumps(response) + "\n").encode())
    conn.close()
    srv.close()

In [10]:
HOST = "127.0.0.1"
PORT = 9700

# start fake daemon
threading.Thread(
    target=fake_daemon,
    args=(HOST, PORT),
    daemon=True
).start()

time.sleep(0.5)

fernet = Fernet(Fernet.generate_key())

In [11]:
process = {
    "client_pkg": {
        "key": "test-key",
        "endpoint": HOST,
        "transport": "daemon",
        "daemon_port": PORT,
        "fernet_key_path": "dummy.key",
    }
}

origin = {"key": "test-key"}

result = send_credential(process, origin)

print("DAEMON RESULT:", result)
assert result is True

Transmission failed: Daemon not responding at 127.0.0.1:9700 — check that credential-daemon.service is running on the target machine


DAEMON RESULT: False


AssertionError: 

In [12]:
process["client_pkg"]["transport"] = "webapp"

result = send_credential(process, origin)

print("WEBAPP RESULT:", result)
assert result is True

send_link called
transmit called
WEBAPP RESULT: True


In [13]:
process["client_pkg"]["transport"] = "ssh"
process["client_pkg"]["username"] = "testuser"
process["client_pkg"]["endpoint"] = "127.0.0.1"

result = send_credential(process, origin)

print("SSH RESULT:", result)
assert result is True

Transmission failed: No module named 'ssh'


SSH RESULT: False


AssertionError: 

In [14]:
bad_origin = {"key": "WRONG"}

result = send_credential(process, bad_origin)

print("FAIL RESULT:", result)
assert result is False

Credential validation failed — aborting transmission


FAIL RESULT: False
